# SPFI ×4 — width check: C8 vs C16 (Colab, CPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/riki/compact-width/notebooks/01_width_colab_c8_c16.ipynb)

**Research question (future):** does widening SPFI from 8 to 16 feature channels (`embed_dim`) improve
×4 super-resolution enough to justify the extra cost?

**What this notebook is:** the *preparation and validation* step only. It checks that C16 differs from
C8 in exactly one setting, that both models build through the repo's own factory (`algo.model.build_spfi`),
run forward/backward on CPU with finite values, produce the right output shape, and have the expected
parameter counts. It records the code version, environment, config hashes and init hashes.

**What it is not:** it does not train anything, does not load trained weights, does not touch the test
set, and does **not** compare quality or speed. A parameter count says nothing about PSNR or runtime.

> Feature channels are the model's *internal capacity* (width of the feature maps inside the network),
> **not** the image resolution. Both C8 and C16 take a 1-channel (grayscale) image and return a
> 1-channel image 4× larger.

**Before you start:** Runtime → Change runtime type → **CPU** is enough. Then Runtime → Run all.

## Settings
`REF` selects which branch (or commit) of the code is tested. Run the notebook once with `REF = "main"`
and once with `REF = "riki/compact-width"`, download both `results_*.json` files, then use the
**branch comparison** section at the end.

In [ ]:
import os
import sys
from pathlib import Path

# ---- Which code to test ----
CODE_REPO_URL = os.environ.get("SR_REPO_URL", "https://github.com/SS1162/SR_amat_3.git")
REF = os.environ.get("SR_REF", "main")       # "main" or "riki/compact-width" (or a commit hash)

# ---- Reproducibility ----
SEED = 0
LR_SIZES = [(32, 32), (24, 40)]   # low-res test inputs (H, W); must be even (DTCWT requirement)

# ---- The two configs. The only intended difference is embed_dim (feature channels). ----
# Kept inline so the notebook can test any branch, including main, which has no configs/ file.
# If the tested branch has configs/width_c8_c16.json, it is checked against these below.
CONFIGS = {
    "c08": dict(scale=4, in_channels=1, embed_dim=8,  n_blocks=4, depth=2, num_heads=4,
                mlp_ratio=1, drop=0.0, attn_drop=0.0, fen_depth=0),
    "c16": dict(scale=4, in_channels=1, embed_dim=16, n_blocks=4, depth=2, num_heads=4,
                mlp_ratio=1, drop=0.0, attn_drop=0.0, fen_depth=0),
}

# Parameter counts recorded on branch riki/compact-width (commit c92fa37), CPU, torch 2.x.
# A mismatch means the model code changed (or a config changed) — investigate before comparing.
EXPECTED_PARAMS = {"c08": 297_313, "c16": 632_617}

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = Path("/content") if IN_COLAB else Path(os.environ.get("SR_WORK_DIR", Path.cwd() / "_width_check"))
CODE_DIR = WORK_DIR / "SR_amat_3"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("REF:", REF, "| Colab:", IN_COLAB, "| work dir:", WORK_DIR)

## Access to the code
The repo is cloned at `REF` and its commit hash is recorded.

**If the repo is private** (authorized students only): create a GitHub *fine-grained personal access
token* with read-only access to this repo, then in Colab open the 🔑 **Secrets** panel (left sidebar),
add a secret named `GITHUB_TOKEN`, and enable notebook access. The cell below reads it from there.
**Never paste a token into a cell** — it would be saved in the notebook and its outputs.
If no secret is set, the clone is anonymous (works while the repo is public).

In [ ]:
import subprocess
import time


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups. Never echoes the URL (it may hold a token)."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout.strip()
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed (exit {result.returncode})")


def repo_url():
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None   # no secret set, or notebook access not granted
    if token and CODE_REPO_URL.startswith("https://"):
        return CODE_REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1), True
    return CODE_REPO_URL, False


url, used_token = repo_url()
if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", url, str(CODE_DIR))
git("remote", "set-url", "origin", url, cwd=CODE_DIR)
try:
    git("fetch", "--quiet", "origin", REF, cwd=CODE_DIR)   # REF = branch name or commit hash
    git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
finally:
    git("remote", "set-url", "origin", CODE_REPO_URL, cwd=CODE_DIR)   # never leave a token in .git/config
del url

COMMIT = git("rev-parse", "HEAD", cwd=CODE_DIR)
COMMIT_INFO = git("log", "-1", "--format=%h %ad %s", "--date=short", cwd=CODE_DIR)
print("token used:", used_token)
print("commit:", COMMIT)
print("       ", COMMIT_INFO)

## Dependencies
torch / numpy are preinstalled on Colab; this adds what `algo/` needs on top (same as `train_colab.ipynb`).

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pytorch_wavelets>=1.3", "PyWavelets>=1.6", "setuptools<81"], check=True)

## Environment and seeds
Everything below runs on CPU with fixed seeds. The exact environment (`pip freeze`) is saved next to the results.

In [ ]:
import hashlib
import json
import platform
import random
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import numpy as np
import torch

sys.path.insert(0, str(CODE_DIR))
from algo.model import build_spfi   # the repo's own factory — no copied model code


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


DEVICE = torch.device("cpu")
torch.use_deterministic_algorithms(True)
set_seeds(SEED)

try:
    from importlib.metadata import version
    PW_VERSION = version("pytorch_wavelets")
except Exception:
    PW_VERSION = "unknown"

ENV = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "processor": platform.processor() or platform.machine(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pytorch_wavelets": PW_VERSION,
    "device": str(DEVICE),
    "torch_threads": torch.get_num_threads(),
    "colab": IN_COLAB,
}
pip_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], text=True, capture_output=True).stdout
(OUTPUT_DIR / "pip_freeze.txt").write_text(pip_freeze)
for k, v in ENV.items():
    print(f"{k:>17}: {v}")

## Teaching example: EDSR-lite (learning material only — not a research result)
A minimal EDSR-style network, written here only to show what "feature channels" means: the image enters
with 1 channel, is lifted to `channels` feature maps, processed, then upsampled ×4 back to 1 channel.
Doubling `channels` roughly quadruples the conv weights, but the input/output shapes do not change.
This toy model is **not** SPFI and its numbers are not used anywhere below.

In [ ]:
import torch.nn as nn


class EDSRLite(nn.Module):
    """Teaching toy: conv -> residual blocks -> 2x PixelShuffle(2) -> conv. NOT the SPFI model."""

    def __init__(self, channels, n_blocks=2):
        super().__init__()
        self.head = nn.Conv2d(1, channels, 3, padding=1)
        self.body = nn.Sequential(*[
            nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(),
                          nn.Conv2d(channels, channels, 3, padding=1))
            for _ in range(n_blocks)])
        self.up = nn.Sequential(
            nn.Conv2d(channels, channels * 4, 3, padding=1), nn.PixelShuffle(2),
            nn.Conv2d(channels, channels * 4, 3, padding=1), nn.PixelShuffle(2))
        self.tail = nn.Conv2d(channels, 1, 3, padding=1)

    def forward(self, x):
        f = self.head(x)
        f = f + self.body(f)
        return self.tail(self.up(f))


x = torch.randn(1, 1, 32, 32)
for ch in (8, 16):
    toy = EDSRLite(ch)
    print(f"EDSR-lite (teaching) channels={ch:>2}: input {tuple(x.shape)} -> output {tuple(toy(x).shape)}, "
          f"params {sum(p.numel() for p in toy.parameters()):,}")

## 1. Exactly one change between C8 and C16
The configs must differ only in `embed_dim`. Scale ×4, block count, depth, heads, MLP ratio, the
shallow feature extractor and the SPM/TMM/CTI structure inside each block are shared.
If the tested branch has `configs/width_c8_c16.json`, it must match the configs above.

In [ ]:
CONFIG_DIFF = {k: (CONFIGS["c08"][k], CONFIGS["c16"][k])
               for k in sorted(set(CONFIGS["c08"]) | set(CONFIGS["c16"]))
               if CONFIGS["c08"].get(k) != CONFIGS["c16"].get(k)}
CHECKS = {}   # name -> bool
CHECKS["only embed_dim differs"] = list(CONFIG_DIFF) == ["embed_dim"]
CHECKS["embed_dim 8 -> 16"] = CONFIG_DIFF.get("embed_dim") == (8, 16)

repo_cfg_path = CODE_DIR / "configs" / "width_c8_c16.json"
if repo_cfg_path.exists():
    repo_cfg = {k: v for k, v in json.loads(repo_cfg_path.read_text()).items() if not k.startswith("_")}
    CHECKS["repo config file matches notebook"] = repo_cfg == CONFIGS
    print("repo config file found and compared:", CHECKS["repo config file matches notebook"])
else:
    print("no configs/width_c8_c16.json on this ref (expected on main) — using the notebook's configs")

print("difference:", CONFIG_DIFF)

## 2. Build both models through the factory and run sanity checks
For each config and each low-res input size: build with a fixed seed, run forward and backward, and check
- output shape is `[1, 1, 4H, 4W]` (grayscale, ×4),
- output, loss and every gradient are finite,
- every parameter receives a gradient,
- parameter count matches the recorded value.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


def build(name):
    set_seeds(SEED)
    return build_spfi(**CONFIGS[name]).to(DEVICE)


ROWS = {}
for name in CONFIGS:
    model = build(name)
    n_params = count_params(model)
    shapes, finite_ok, grads_ok, shape_ok = [], True, True, True
    for (h, w) in LR_SIZES:
        model.zero_grad(set_to_none=True)
        torch.manual_seed(SEED)
        x = torch.rand(1, 1, h, w, device=DEVICE)
        y = model(x)
        loss = (y - torch.nn.functional.interpolate(x, scale_factor=4, mode="bicubic")).abs().mean()
        loss.backward()
        shape_ok &= tuple(y.shape) == (1, 1, 4 * h, 4 * w)
        finite_ok &= bool(torch.isfinite(y).all()) and bool(torch.isfinite(loss))
        grads = [p.grad for p in model.parameters() if p.requires_grad]
        grads_ok &= all(g is not None for g in grads) and all(bool(torch.isfinite(g).all()) for g in grads if g is not None)
        shapes.append(f"{tuple(x.shape)} -> {tuple(y.shape)}")
    ROWS[name] = {
        "embed_dim": CONFIGS[name]["embed_dim"],
        "params": n_params,
        "expected_params": EXPECTED_PARAMS[name],
        "shapes": shapes,
        "checks": {
            "output shape [1,1,4H,4W]": shape_ok,
            "forward finite": finite_ok,
            "backward finite, all params get grads": grads_ok,
            "params == expected": n_params == EXPECTED_PARAMS[name],
        },
    }
    for k, v in ROWS[name]["checks"].items():
        CHECKS[f"{name}: {k}"] = v
    print(f"{name}: {n_params:,} params | " + " ; ".join(shapes))

CHECKS["c16 has more params than c08"] = ROWS["c16"]["params"] > ROWS["c08"]["params"]

## 3. Hashes: configs and initialization
- **Config hash**: SHA-256 of the config as canonical JSON.
- **Init hash**: SHA-256 of all initial weights (in `state_dict` order) after seeding with `SEED`.
  Building the same config twice must give the same init hash (reproducible init).

C8 and C16 have different tensor shapes, so their init hashes are *expected* to differ — this notebook
does **not** claim any shared or identical tensors between them, and no trained C8 weights are loaded.

In [ ]:
def config_hash(cfg):
    return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()


def init_hash(model):
    h = hashlib.sha256()
    for key, t in model.state_dict().items():
        h.update(key.encode())
        h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


for name in CONFIGS:
    h1, h2 = init_hash(build(name)), init_hash(build(name))
    ROWS[name]["config_sha256"] = config_hash(CONFIGS[name])
    ROWS[name]["init_sha256"] = h1
    ROWS[name]["checks"]["init reproducible"] = h1 == h2
    CHECKS[f"{name}: init reproducible"] = h1 == h2
    print(f"{name}: config {ROWS[name]['config_sha256'][:16]}…  init {h1[:16]}…  reproducible={h1 == h2}")

## 4. Results table

In [ ]:
import pandas as pd

table = pd.DataFrame([{
    "model": name.upper(),
    "channels (embed_dim)": r["embed_dim"],
    "params": f"{r['params']:,}",
    "input -> output": " ; ".join(r["shapes"]),
    "config diff vs C08": "—" if name == "c08" else ", ".join(f"{k}: {a} → {b}" for k, (a, b) in CONFIG_DIFF.items()),
    "PASS/FAIL": "PASS" if all(r["checks"].values()) else "FAIL",
} for name, r in ROWS.items()])

checks_table = pd.DataFrame([{"check": k, "result": "PASS" if v else "FAIL"} for k, v in CHECKS.items()])
ALL_PASS = all(CHECKS.values())

pd.set_option("display.max_colwidth", None)
display(table)
display(checks_table)
print("ALL CHECKS:", "PASS" if ALL_PASS else "FAIL")

## 5. Parameter count
Shown for size only. More parameters does **not** imply better quality or a given speed — those are
measured later (see the measurement plan).

In [ ]:
import matplotlib.pyplot as plt

names = [n.upper() for n in ROWS]
params = [ROWS[n]["params"] for n in ROWS]
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(names, params, color=["#4C72B0", "#DD8452"], width=0.5)
ax.bar_label(bars, labels=[f"{p:,}" for p in params], padding=3)
ax.set_ylabel("trainable parameters")
ax.set_title(f"SPFI ×4 parameter count — {REF} @ {COMMIT[:7]}")
ax.set_ylim(0, max(params) * 1.18)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "params_bar.png", dpi=120)
plt.show()
print(f"C16 / C08 parameter ratio: {params[1] / params[0]:.2f}×")

## 6. Save results
Saves `results_<ref>.json` (code version, environment, configs, hashes, checks) to the output folder and,
on Colab, downloads it. Keep these files and the executed notebook **outside git**.

In [ ]:
safe_ref = REF.replace("/", "_")
RESULTS = {
    "ref": REF,
    "commit": COMMIT,
    "commit_info": COMMIT_INFO,
    "seed": SEED,
    "lr_sizes": LR_SIZES,
    "env": ENV,
    "configs": CONFIGS,
    "config_diff": {k: list(v) for k, v in CONFIG_DIFF.items()},
    "models": ROWS,
    "checks": CHECKS,
    "all_pass": ALL_PASS,
}
results_path = OUTPUT_DIR / f"results_{safe_ref}.json"
results_path.write_text(json.dumps(RESULTS, indent=2))
print("saved", results_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(results_path))

## 7. Branch comparison (main vs riki/compact-width)
Put both `results_*.json` files in the output folder (on Colab: the cell asks you to upload the one from
the other run). Parameter counts, config hashes and init hashes should be **identical** across branches
— that shows the branch changed nothing in the model. Commits will differ, which is expected.

In [ ]:
COMPARE_REFS = ["main", "riki/compact-width"]
paths = {r: OUTPUT_DIR / f"results_{r.replace('/', '_')}.json" for r in COMPARE_REFS}

missing = [r for r, p in paths.items() if not p.exists()]
if missing and IN_COLAB:
    from google.colab import files
    print("Upload:", ", ".join(paths[r].name for r in missing))
    for fname, data in files.upload().items():
        (OUTPUT_DIR / fname).write_bytes(data)
    missing = [r for r, p in paths.items() if not p.exists()]

if missing:
    print("Branch comparison skipped — missing:", ", ".join(paths[r].name for r in missing))
else:
    a, b = (json.loads(paths[r].read_text()) for r in COMPARE_REFS)
    rows = [{"item": "commit", COMPARE_REFS[0]: a["commit"][:7], COMPARE_REFS[1]: b["commit"][:7], "same?": "(differs, expected)"}]
    for name in CONFIGS:
        for key in ("params", "config_sha256", "init_sha256"):
            va, vb = a["models"][name][key], b["models"][name][key]
            short = (lambda v: f"{v:,}" if isinstance(v, int) else v[:16] + "…")
            rows.append({"item": f"{name} {key}", COMPARE_REFS[0]: short(va), COMPARE_REFS[1]: short(vb),
                         "same?": "✅" if va == vb else "❌"})
    rows.append({"item": "all checks", COMPARE_REFS[0]: "PASS" if a["all_pass"] else "FAIL",
                 COMPARE_REFS[1]: "PASS" if b["all_pass"] else "FAIL", "same?": ""})
    display(pd.DataFrame(rows))

## 8. Trained comparison — ⏳ pending
Quality comparison of trained C8 vs C16 will be added **only** once approved checkpoints exist
(per issue #156). Nothing is trained or loaded here; C8 is not retrained and the old 300-step demo is not used.

| model | checkpoint | PSNR | SSIM | runtime | peak memory |
|---|---|---|---|---|---|
| C08 | pending (#156) | — | — | — | — |
| C16 | pending (#156) | — | — | — | — |

## 9. Measurement plan for the future comparison
Run only after approved checkpoints exist; GPU measurements come later.

| what | how | notes |
|---|---|---|
| **PSNR / SSIM** | on the validation split, Y/grayscale, same crop/border for both models | test split stays locked until the final evaluation |
| **Runtime** | median of N forward passes after warm-up, fixed input size, `torch.cuda.synchronize()` | same GPU, batch size and precision for both |
| **Memory** | `torch.cuda.max_memory_allocated()` for inference and for one training step | reset stats before each measurement |
| **Training cost** | wall-clock time and iterations to the best validation checkpoint, GPU type | same data, seed, schedule and early-stopping rule |
| **Decision** | is the PSNR/SSIM gain worth the extra time/memory? | report all numbers; no conclusion from parameter count alone |

## Summary

In [ ]:
passed = [k for k, v in CHECKS.items() if v]
failed = [k for k, v in CHECKS.items() if not v]
print(f"Code: {REF} @ {COMMIT[:7]} ({COMMIT_INFO})")
print(f"What changed: {', '.join(f'{k} {a} -> {b}' for k, (a, b) in CONFIG_DIFF.items())} — nothing else.")
print(f"Passed ({len(passed)}/{len(CHECKS)}):")
for k in passed:
    print("   ✓", k)
print(f"Failed ({len(failed)}):", "none" if not failed else "")
for k in failed:
    print("   ✗", k)
print("Measured: parameter counts "
      + ", ".join(f"{n.upper()}={r['params']:,}" for n, r in ROWS.items())
      + f"; output shapes for LR sizes {LR_SIZES}; config/init hashes (CPU, seed {SEED}).")
print("Not measured: quality (PSNR/SSIM), runtime, memory, training cost — pending (#156).")

**Limitation:** all checks run on CPU with randomly initialized weights. They show the C16 variant is
well-formed and differs from C8 only in width — they say nothing about whether C16 reconstructs images
better or is worth its cost. Init hashes are tied to the torch version and may differ on another setup.